# Patel, Kim et al. (2026) from dandiset 001454: calcium imaging in the cued two-bottle choice

Paper. Patel RR, Kim KN, Patarino M, et al. (2026). Social isolation recruits
amygdala–medial prefrontal cortex projections to escalate alcohol drinking in male mice.
*Nature Neuroscience*. [doi:10.1038/s41593-026-02413-x](https://doi.org/10.1038/s41593-026-02413-x).
The claims file was written from the version of record (PMC13633117), its legends, Methods and
reporting summary.

Dandiset. [DANDI:001454](https://dandiarchive.org/dandiset/001454/0.260911.2319), version
0.260911.2319, the only published version (the paper cites the dandiset without a version; the
release notes say this version holds all data used in the publication; the draft has the same
assets but can change).

This notebook tests the imaging claims of the cued two-bottle choice (Figs. 3, 5 and 6, Extended
Data Figs. 5, 6 and 8). The patch-clamp claims are in `ephys_patel_2026.ipynb`, with the full
table of blocked claims. Each claim's section seeds its own generator.

| Claim | Status |
|---|---|
| ed5d-cue-response-modest | tested |
| fig3f-blampfc-distinct-clusters | tested |
| ed5ef-alcohol-water-nonoverlapping | tested |
| fig3h-nonspecific-decoding-85, fig3h-blampfc-decodes-better, ed6b-decoding-above-chance | blocked: too few bouts under the decoding rule (counts below) |
| fig3k-blampfc-response-predicts-bouts | tested |
| fig3ijl-no-other-correlations | tested |
| ed6ij-cue-responses-no-correlation | tested |
| ed6k-bout-duration-not-number | tested |
| fig5b, fig5c, fig5d (housing) | tested |
| fig6c, fig6de, fig6f, fig6gj, fig6k, fig6lm, fig6n, ed8n, ed8op (stimulation) | tested |

Which population is which. The BLA files do not say whether GCaMP was expressed in all BLA
neurons or only in BLA-mPFC projectors (the virus field is empty). Cohorts 9 and 10 hold 10 mice
and cohort 16 holds 11, the paper's N for nonspecific BLA (10) and BLA-mPFC (11) imaging; cohort 16
also has fewer ROIs per mouse, as projection-restricted expression would give. Cohorts 9 and 10 are
taken as nonspecific BLA and cohort 16 as BLA-mPFC: an assumption, marked on every claim that
uses it. The mPFC files (cohort 13b, location "PrL") are the stimulation sessions.

Neurons. The files hold every CNMF-E ROI. The paper's neurons were "visually curated by
experimenters blinded to experimental condition; neurons with abnormal morphology or calcium
traces were excluded" (Methods), and the curation is not in the dandiset. All ROIs are used.

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
from scipy import stats
from scipy.cluster.hierarchy import fcluster, linkage

import patel2026 as P

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 200)
t_start = time.time()
print("dandiset", P.DANDISET_ID, "version", P.DANDISET_VERSION)
A = P.assets()
op = A[A.path.str.endswith("behavior+image+ophys.nwb")].reset_index(drop=True)
t0 = time.time()
S = P.parallel(P.load_session, list(op.asset_id), workers=26)
for s, p in zip(S, op.path):
    s["path"] = p
    s["cohort"] = s["subject"].split("-")[0]
print(f"loaded {len(S)} sessions in {time.time() - t0:.0f} s")

## Data check

In [ ]:
rows = []
for s in S:
    L = s["licks"]
    rows.append(dict(cohort=s["cohort"], subject=s["subject"], description=s["description"], location=s["location"],
                     frames=s["n_frames"], minutes=round(s["n_frames"] / s["rate"] / 60, 1), rate=s["rate"], rois=s["n_rois"],
                     reward_licks=len(L["reward"]), water_licks=len(L["water"]),
                     last_lick_s=round(max(np.r_[L["reward"], L["water"], 0]), 0), cues=len(s["cue_t"]),
                     types="".join(map(str, s["cue_type"]))))
T = pd.DataFrame(rows)
print(T.drop(columns="types").to_string())
print("\nimaging rate (Hz):", T.rate.unique(), "; ROI trace unit a.u.")
print("lick onsets before offsets in every session:",
      all(np.all(s["licks"]["reward_end"] >= s["licks"]["reward"]) and np.all(s["licks"]["water_end"] >= s["licks"]["water"]) for s in S))
print("distinct cue time vectors:", len({tuple(s["cue_t"]) for s in S}), "; distinct trial-type sequences:", T.types.nunique())
print(T.groupby("types").subject.apply(lambda x: sorted(set(x))).to_dict())

In [ ]:
# The same imaging deposited twice?
import hashlib
hsh = pd.Series([hashlib.md5(s["F"].tobytes()).hexdigest() for s in S])
dup = hsh[hsh.duplicated(keep=False)]
for h in dup.unique():
    idx = list(dup[dup == h].index)
    print("identical traces:", [S[i]["path"].split("/")[1][4:45] for i in idx])
print("identical lick vectors:", int(pd.Series([hashlib.md5(np.r_[s['licks']['reward'], s['licks']['water']].tobytes()).hexdigest() for s in S]).duplicated().sum()))

# Licks against the cue schedule: share of licks in the 58-s access window (cue onset + 3 s to + 61 s),
# and licks of the bottle that a forced trial does not offer
def schedule_check(s):
    cue, L = s["cue_t"], np.r_[s["licks"]["reward"], s["licks"]["water"]]
    i = np.searchsorted(cue, L, side="right") - 1
    rel = L - cue[np.clip(i, 0, None)]
    out = dict(in_access=np.mean((i >= 0) & (rel >= 3) & (rel <= 61)) if len(L) else np.nan,
               in_cue=np.mean((i >= 0) & (rel < 3)) if len(L) else np.nan)
    for liq, bad in (("water", 2), ("reward", 1)):
        x = s["licks"][liq]; j = np.searchsorted(cue, x, side="right") - 1
        out[f"{liq}_in_forced_other"] = int(np.sum((j >= 0) & (s["cue_type"][np.clip(j, 0, None)] == bad)))
    return out
SC = pd.DataFrame([schedule_check(s) for s in S], index=[s["path"].split("/")[1][4:45] for s in S])
SC["types_match_licks"] = (SC.water_in_forced_other == 0) & (SC.reward_in_forced_other == 0)
print(SC.round(3).to_string())

Every session carries the same cue times, and one of two trial-type sequences. The licks fall in
the access windows of those cues (in_access near 1, no lick during the 3-s cue), so the cue times are
consistent with the licks. But in many sessions there are licks on the bottle that a forced trial,
by its type, does not offer: there the trial types (and so which choice trials were stimulated) do
not match the session. The stimulation claims are tested with the types as stored, and again on the
sessions whose forced trials match the licks.

Three pairs of sessions hold identical traces with different licks. The next cell asks whose licks
the traces follow.

In [ ]:
# Clock check: the population response at bout onsets (median over neurons of |mean z, 0 to 3 s|),
# with the true onsets and with onsets shifted by -20 to +20 s
lags = np.arange(-20, 21, 1.0)
def strength(s, onsets):
    z = P.event_z(s["F"], onsets, post_s=3.0)
    if len(z) == 0:
        return np.nan
    return float(np.nanmedian(np.abs(np.nanmean(P.window_mean(z), axis=0))))
CK = []
for s in S:
    b = P.session_bouts(s); on = b.onset.values[b.isolated.values]
    st = np.array([strength(s, on + l) for l in lags])
    far = np.nanmedian(st[np.abs(lags) >= 6])
    CK.append(dict(session=s["path"].split("/")[1][4:45], bouts=len(on), lag0=st[20], far=far, best_lag=lags[np.nanargmax(st)]))
CK = pd.DataFrame(CK)
CK["aligned"] = (CK.lag0 > CK.far) & (CK.best_lag.between(-2, 5))
print(CK.round(3).to_string())
print("sessions where alignment is shown:", int(CK.aligned.sum()), "of", len(CK))
# duplicated traces: whose licks do they follow?
pairs = [("Cohort16-M1-4", "Cohort16-M2-1"), ("Cohort16-M3-3", "Cohort16-M3-4"),
         ("Cohort13b-M2-2_ses-20230218", "Cohort13b-M3-2_ses-20230126")]
for a, b_ in pairs:
    sa = [s for s in S if a in s["path"]][0]; sb = [s for s in S if b_ in s["path"]][0]
    for lab, lk in ((a, sa), (b_, sb)):
        bb = P.session_bouts(lk); on = bb.onset.values[bb.isolated.values]
        sh = np.nanmedian([strength(sa, on + l) for l in (-15, -10, 10, 15)])
        print(f"traces of the pair with the licks of {lab}: {strength(sa, on):.3f} at the true onsets, {sh:.3f} shifted")

The traces of each identical pair follow the licks of one member: cohort 16 M1-4 (0.339 against
0.191 shifted) and M3-3, and cohort 13b M2-2's isolation sucrose session. The other member's file
(cohort 16 M2-1 and M3-4, and cohort 13b M3-2's isolation sucrose session) carries another
mouse's imaging and is left out. The clock is shown for the sessions marked aligned above; the
others have few bouts or no peak near lag 0, and are named in the summary.

In [ ]:
EXCLUDE = ("Cohort16-M2-1_ses", "Cohort16-M3-4_ses", "Cohort13b-M3-2_ses-20230126")
SS = [s for s in S if not any(e in s["path"] for e in EXCLUDE)]
NS = [s for s in SS if s["cohort"] in ("Cohort9", "Cohort10")]
PR = [s for s in SS if s["cohort"] == "Cohort16"]
MP = [s for s in SS if s["cohort"] == "Cohort13b"]
def kind(s):
    d = s["description"]
    return ("GH" if d.startswith("Pre") else "SI") + "_" + ("alcohol" if "alcohol" in d else "sucrose")
for s in MP:
    s["kind"] = kind(s)
for lab, grp in (("nonspecific BLA (cohorts 9, 10)", NS), ("BLA-mPFC (cohort 16)", PR)):
    print(f"{lab}: {len(grp)} mice, {sum(s['n_rois'] for s in grp)} ROIs "
          f"(per mouse {[s['n_rois'] for s in grp]}); alcohol licks per mouse {[len(s['licks']['reward']) for s in grp]}")
print("paper: nonspecific 540 neurons from 10 mice, BLA-mPFC 196 from 11")
for k in ("GH_alcohol", "SI_alcohol", "GH_sucrose", "SI_sucrose"):
    g = [s for s in MP if s["kind"] == k]
    print(f"mPFC {k}: {len(g)} mice, {sum(s['n_rois'] for s in g)} ROIs; reward-bottle licks {[(s['subject'][-4:], len(s['licks']['reward'])) for s in g]}")
print("paper: alcohol 240 neurons from 6 mice (GH), 281 from 5 (SI); sucrose 284 from 7, 343 from 7")

The rule that mice with fewer than 50 alcohol licks are excluded from calcium analyses (Methods,
2BC drinking) gives exactly the paper's 6 group-housed and 5 isolated mice for the mPFC alcohol
sessions, and is applied there. For BLA imaging the paper's N (10 and 11) equals every mouse in
the dandiset, although Cohort9-M5-2 (44 licks) and Cohort16-M1-4 (49) are under 50, so the rule
is not applied there (shown as a variant where it bears on a correlation). For the sucrose
sessions the rule names no session; all mice with a session are used (8 group-housed, where the
paper has 7; 6 isolated after the duplicate, where it has 7).

In [ ]:
MP_A_GH = [s for s in MP if s["kind"] == "GH_alcohol" and len(s["licks"]["reward"]) >= P.MIN_ALCOHOL_LICKS]
MP_A_SI = [s for s in MP if s["kind"] == "SI_alcohol" and len(s["licks"]["reward"]) >= P.MIN_ALCOHOL_LICKS]
MP_S_GH = [s for s in MP if s["kind"] == "GH_sucrose"]
MP_S_SI = [s for s in MP if s["kind"] == "SI_sucrose"]
for lab, g in (("alcohol GH", MP_A_GH), ("alcohol SI", MP_A_SI), ("sucrose GH", MP_S_GH), ("sucrose SI", MP_S_SI)):
    print(lab, [s["subject"] for s in g], sum(s["n_rois"] for s in g), "ROIs")
# isolated bouts and trial types per session
def bout_counts(s):
    b = P.session_bouts(s); b = b[b.isolated]
    return {f"{liq}_{t}": int(((b.liquid == liq) & (b.trial_type == t)).sum()) for liq in ("reward", "water") for t in (1, 2, 3, 4)}
BC = pd.DataFrame([bout_counts(s) for s in SS], index=[s["path"].split("/")[1][4:45] for s in SS])
print(BC.to_string())
# z baselines removed by the s.d. floor
tot = rem = 0
for s in SS:
    b = P.session_bouts(s); on = b.onset.values[b.isolated.values]
    z = P.event_z(s["F"], on, post_s=3.0)
    tot += z[..., 0].size; rem += int(np.isnan(z[..., 0]).sum())
print(f"bout-neuron pairs with a baseline s.d. under {P.SD_FLOOR} of the session s.d.: {rem} of {tot} ({rem / tot:.1%})")

## ed5d-cue-response-modest

> Both BLA populations respond to the cue with a modest, transient increase below 0.5 z-score.

Criterion: cue-onset z-scores per M3 with cue onset in place of bout onset, baselines 3 s and 5 s;
reproduced, under both readings, when the population mean trace of each population peaks above 0
and below 0.5 within 3 s of cue onset and falls back toward 0 by the end of the 3-s cue. All 30
cues are used (the claim names no trial subset). "Falls back toward 0" is read as the trace at
3 s being below its peak.

In [ ]:
def cue_trace(grp, pre):
    tr = [np.nanmean(P.event_z(s["F"], s["cue_t"], pre_s=pre, post_s=5.0), axis=0) for s in grp]  # neurons x frames
    return np.concatenate(tr, axis=0)
ED5D = {}
for pre in (3.0, 5.0):
    for lab, grp in (("nonspecific", NS), ("BLA-mPFC", PR)):
        tr = cue_trace(grp, pre)
        m = np.nanmean(tr, axis=0)
        t = (np.arange(m.size) - int(pre * 10)) / 10
        w = (t >= 0) & (t <= 3)
        pk = np.nanmax(m[w]); tpk = t[w][np.nanargmax(m[w])]
        end = m[np.argmin(np.abs(t - 3.0))]
        ED5D[(pre, lab)] = (pk, end)
        print(f"baseline {pre:.0f} s, {lab}: {tr.shape[0]} neurons; mean z peak {pk:.3f} at {tpk:.1f} s, at 3 s {end:.3f}, mean 0-3 s {np.nanmean(m[w]):.3f}; "
              f"trace range {np.nanmin(m):.3f} to {np.nanmax(m):.3f}")
        if pre == 3.0:
            per = [np.nanmax(np.nanmean(np.nanmean(P.event_z(s['F'], s['cue_t'], pre_s=pre, post_s=5.0), axis=0), axis=0)[30:61]) for s in grp]
            print("   peak 0-3 s by mouse:", np.round(per, 3).tolist())

**Result: not reproduced, resting on the assignment of cohorts to populations.** For the BLA-mPFC population (cohort 16) the mean cue trace stays small and positive (peak 0.106 with a 3-s baseline, 0.118 with 5 s, below 0.5) but rises through the cue and peaks at 2.9 s, so it does not fall back by the end of the cue. For the nonspecific population (cohorts 9 and 10) the mean trace after the cue is negative under both baselines (largest value within 3 s, -0.520 and -0.412): the four cohort-10 mice peak at -0.003 to 0.082, the six cohort-9 mice at -0.87 to -0.29. Neither population shows the transient increase.**

## fig3f-blampfc-distinct-clusters

> Hierarchical clustering of alcohol responses gives several functional clusters, and BLA-mPFC
> neurons distribute across them differently from nonspecific BLA neurons.

Criterion: one clustering of both populations pooled; Ward linkage, Euclidean distance on each
neuron's mean alcohol z trace (M3), cut at the highest linkage giving at least one nonresponsive
cluster (a cluster whose mean trace stays within ±1.98 after onset); trace spans -3 to +5 s and 0
to +3 s; chi-square of population by cluster, p < 0.05 under both spans. Mean traces use alcohol
and water bouts matched per session (M3 reading a, random); neurons without a defined trace (no
isolated alcohol bout) are left out.

In [ ]:
rng = np.random.default_rng(31)
def alcohol_water_traces(grp, rng, how="random"):
    out = {"reward": [], "water": [], "mouse": [], "n": []}
    for s in grp:
        b = P.session_bouts(s)
        tr, m, _ = P.condition_traces(s, b, {"reward": (b.liquid == "reward").values, "water": (b.liquid == "water").values}, rng, how)
        out["reward"].append(tr["reward"]); out["water"].append(tr["water"])
        out["mouse"] += [s["subject"]] * s["n_rois"]; out["n"].append(m)
    return {k: (np.concatenate(v) if k in ("reward", "water") else v) for k, v in out.items()}
TR = {lab: alcohol_water_traces(g, rng) for lab, g in (("nonspecific", NS), ("BLA-mPFC", PR))}
for lab in TR:
    print(lab, "matched alcohol/water bouts per mouse:", TR[lab]["n"])

In [ ]:
def cluster_test(span):
    X = np.concatenate([TR["nonspecific"]["reward"], TR["BLA-mPFC"]["reward"]])
    pop = np.array(["nonspecific"] * len(TR["nonspecific"]["reward"]) + ["BLA-mPFC"] * len(TR["BLA-mPFC"]["reward"]))
    t = (np.arange(X.shape[1]) - 30) / 10
    cols = (t >= -3) & (t < 5) if span == "-3 to 5 s" else (t >= 0) & (t < 3)
    keep = np.all(np.isfinite(X[:, cols]), axis=1)
    Xk, popk = X[keep][:, cols], pop[keep]
    Z = linkage(Xk, method="ward", metric="euclidean")
    after = t[cols] >= 0
    for k in range(2, 60):
        lab = fcluster(Z, k, criterion="maxclust")
        nonresp = [c for c in np.unique(lab) if np.all(np.abs(Xk[lab == c].mean(0)[after]) < P.Z_THRESHOLD)]
        if nonresp:
            break
    tab = pd.crosstab(lab, popk)
    chi = stats.chi2_contingency(tab)
    exp = tab.sum(1).values[:, None] * tab.sum(0).values[None, :] / tab.values.sum()
    enrich = (tab.values - exp) / exp * 100
    print(f"span {span}: {keep.sum()} of {len(X)} neurons clustered; first cut with a nonresponsive cluster: {k} clusters (nonresponsive {nonresp})")
    print(tab.assign(**{f"enrich_{c}_%": np.round(enrich[:, i], 1) for i, c in enumerate(tab.columns)}).to_string())
    print(f"  chi-square({chi.dof}) = {chi.statistic:.2f}, p = {chi.pvalue:.3g}")
    return chi.pvalue
FIG3F = {span: cluster_test(span) for span in ("-3 to 5 s", "0 to 3 s")}

**Result: reproduced, resting on the assignment of cohorts to populations and on all ROIs standing for the curated neurons.** Under the paper's cut (the highest linkage giving a nonresponsive cluster) the pooled clustering of 1266 neurons gives 2 clusters under both spans, and the population by cluster test is significant (chi-square(1) = 18.30, p = 1.9e-05, for -3 to 5 s; 11.20, p = 0.00082, for 0 to 3 s). The difference lies in the small responsive cluster: it holds 66 (or 43) nonspecific neurons and 3 (or 2) BLA-mPFC neurons, a BLA-mPFC depletion of 84%. The cut does not give the several functional clusters the statement describes.**

## ed5ef-alcohol-water-nonoverlapping

> Neurons excited by alcohol and those excited by water are largely separate ensembles.

Criterion: excited neurons per M4 for alcohol and for water bouts, in each population (the
Wilcoxon reading shown too); reproduced when fewer than half of the smaller excited set is also
excited by the other liquid, in both populations, under the threshold reading. Responses use the
matched traces of fig3f (reading a) and the first-bouts reading (b).

In [ ]:
rng = np.random.default_rng(55)
TRb = {lab: alcohol_water_traces(g, rng, how="first") for lab, g in (("nonspecific", NS), ("BLA-mPFC", PR))}
def overlap(ra, rw):
    ea, ew = ra >= P.Z_THRESHOLD, rw >= P.Z_THRESHOLD
    both = int((ea & ew).sum()); small = min(ea.sum(), ew.sum())
    return int(ea.sum()), int(ew.sum()), both, (both / small if small else np.nan)
ED5EF = {}
for reading, D_ in (("a random", TR), ("b first", TRb)):
    for lab in D_:
        ra, rw = P.window_mean(D_[lab]["reward"]), P.window_mean(D_[lab]["water"])
        ok = np.isfinite(ra) & np.isfinite(rw)
        ea, ew, both, frac = overlap(ra[ok], rw[ok])
        ED5EF[(reading, lab)] = frac
        print(f"reading {reading}, {lab}: {ok.sum()} neurons with both responses; excited alcohol {ea}, water {ew}, both {both}; overlap / smaller set {frac:.2f}")
# Wilcoxon reading over all isolated bouts of each liquid
for lab, grp in (("nonspecific", NS), ("BLA-mPFC", PR)):
    ea = ew = both = 0
    for s in grp:
        b = P.session_bouts(s); b = b[b.isolated]
        za = P.event_z(s["F"], b.onset.values[b.liquid == "reward"], post_s=3.0)
        zw = P.event_z(s["F"], b.onset.values[b.liquid == "water"], post_s=3.0)
        if len(za) < 5 or len(zw) < 5:
            continue
        wa, ww = P.wilcoxon_responsive(za) == 1, P.wilcoxon_responsive(zw) == 1
        ea += wa.sum(); ew += ww.sum(); both += (wa & ww).sum()
    print(f"Wilcoxon reading, {lab} (mice with >= 5 bouts of each): excited alcohol {ea}, water {ew}, both {both}, overlap / smaller {both / max(1, min(ea, ew)):.2f}")

**Result: reproduced, resting on the assignment of cohorts to populations.** Few neurons are excited by both liquids: in nonspecific BLA 9 of the smaller set of 52 (0.17; reading b 0.12), in BLA-mPFC 1 of 5 (0.20; reading b 0.00). The Wilcoxon reading gives 0.17 and 0.26. The BLA-mPFC sets are small (5 to 8 neurons under the threshold).**

## Decoding claims (fig3h-nonspecific-decoding-85, fig3h-blampfc-decodes-better, ed6b-decoding-above-chance): blocked

M5 uses bouts longer than 1 s separated from other bouts by at least 2 s, with five folds per
animal, each holding at least one bout of each class, so an animal needs five such bouts of
alcohol and five of water. The counts per mouse:

In [ ]:
DC = []
for lab, grp in (("nonspecific", NS), ("BLA-mPFC", PR)):
    for s in grp:
        b = P.decoding_bouts(s)
        DC.append(dict(population=lab, mouse=s["subject"], alcohol=int((b.liquid == "reward").sum()), water=int((b.liquid == "water").sum())))
DC = pd.DataFrame(DC)
DC["usable"] = (DC.alcohol >= 5) & (DC.water >= 5)
print(DC.to_string(index=False))
print(DC.groupby("population").usable.sum().to_dict(), "mice have five or more bouts of each class")

Only 1 of 10 nonspecific mice and 2 of 9 BLA-mPFC mice have five decoding bouts of each liquid;
most mice have 0 to 2 water bouts. The paper's procedure cannot be run on these bouts, so the
three decoding claims are blocked (`other`: too few bouts under the claim's own rule).

## fig3k-blampfc-response-predicts-bouts

> Across mice, the mean BLA-mPFC response to alcohol correlates positively with the number of
> alcohol bouts.

Criterion: per mouse, the mean over its BLA-mPFC neurons of the M3 alcohol response, against its
number of alcohol bouts (M1) in the imaging session; Pearson; Bonferroni over (a) four tests (Fig.
3i to l) or (b) two tests (k and l); reproduced when r2 = 0.59 ± 0.059, r positive and corrected
two-sided p < 0.05 under both. Bouts counted are all M1 alcohol bouts.

In [ ]:
def per_mouse(lab, liquid, how_tr=TR):
    tr = how_tr[lab]
    r = P.window_mean(tr[liquid]); mice = np.array(tr["mouse"])
    grp = NS if lab == "nonspecific" else PR
    rows = []
    for s in grp:
        rows.append(dict(mouse=s["subject"], response=np.nanmean(r[mice == s["subject"]]),
                         bouts=len(P.bouts(s["licks"][liquid])), licks=len(s["licks"]["reward"])))
    return pd.DataFrame(rows)
def corr(df, name, m_list=(4, 2), excl=False):
    d = df.dropna()
    if excl:
        d = d[d.licks >= P.MIN_ALCOHOL_LICKS]
    r, p = stats.pearsonr(d.response, d.bouts)
    print(f"{name}: n = {len(d)} mice, r = {r:.3f}, r2 = {r * r:.3f}, p = {p:.3g}; Bonferroni x4 {min(1, 4 * p):.3g}, x2 {min(1, 2 * p):.3g}")
    return r, p, len(d)
K = {}
for reading, D_ in (("a random", TR), ("b first", TRb)):
    df = per_mouse("BLA-mPFC", "reward", D_)
    if reading == "a random":
        print(df.round(3).to_string(index=False))
    K[reading] = corr(df, f"fig3k BLA-mPFC alcohol, reading {reading}")
    corr(df, f"   variant without mice under 50 alcohol licks, reading {reading}", excl=True)

**Result: not reproduced, resting on the assignment of cohorts to populations.** Across the 9 BLA-mPFC mice the mean alcohol response and the number of alcohol bouts are weakly and not significantly correlated (r = 0.288, r2 = 0.083, p = 0.45; reading b r2 = 0.015, p = 0.75), against the paper's r2 = 0.59. Leaving out the mouse under 50 licks changes little (r2 = 0.109).**

## fig3ijl-no-other-correlations

> Nonspecific BLA responses do not correlate with alcohol or water bouts, and BLA-mPFC water
> responses do not correlate with water bouts.

Criterion: a claim of absence on the paper's test (Pearson, Bonferroni over Fig. 3i to l) at p <
0.05; reproduced when none of the three reaches corrected p < 0.05.

In [ ]:
IJL = {}
for reading, D_ in (("a random", TR), ("b first", TRb)):
    for lab, liq in (("nonspecific", "reward"), ("nonspecific", "water"), ("BLA-mPFC", "water")):
        df = per_mouse(lab, liq, D_)
        IJL[(reading, lab, liq)] = corr(df, f"{lab} {liq} response against {liq} bouts, reading {reading}")

**Result: reproduced.** None of the three correlations reaches p < 0.05 even before correction: nonspecific alcohol r2 = 0.009 (p = 0.79), nonspecific water r2 = 0.252 (p = 0.14), BLA-mPFC water r2 = 0.052 (p = 0.56); reading b gives the same (smallest p 0.15).**

## ed6ij-cue-responses-no-correlation

> Cue-evoked responses of neither population correlate with alcohol bouts.

Criterion: a claim of absence on the paper's test (Pearson, p < 0.05); per mouse mean cue response
as in ed5d (mean z 0 to 3 s after cue onset) against alcohol bouts; reproduced when neither
correlation has p < 0.05. Both cue baselines (3 s and 5 s) are shown.

In [ ]:
ED6IJ = {}
for pre in (3.0, 5.0):
    for lab, grp in (("nonspecific", NS), ("BLA-mPFC", PR)):
        resp = [np.nanmean(P.window_mean(P.event_z(s["F"], s["cue_t"], pre_s=pre, post_s=5.0), pre_s=pre)) for s in grp]
        nb = [len(P.bouts(s["licks"]["reward"])) for s in grp]
        r, p = stats.pearsonr(resp, nb)
        ED6IJ[(pre, lab)] = p
        print(f"baseline {pre:.0f} s, {lab}: n = {len(grp)}, r = {r:.3f}, r2 = {r * r:.3f}, p = {p:.3g}")

**Result: not reproduced, resting on the assignment of cohorts to populations.** Nonspecific cue responses do not correlate with alcohol bouts (r2 = 0.016, p = 0.73; 5-s baseline p = 0.88), but BLA-mPFC cue responses do (r = 0.733, r2 = 0.537, p = 0.025; 5-s baseline r2 = 0.567, p = 0.019), across 9 mice.**

## ed6k-bout-duration-not-number

> Average bout duration does not correlate with the number of bouts.

Criterion: a claim of absence on the paper's test (Pearson, p < 0.05), alcohol bouts (M1) of the
BLA-mPFC mice; reproduced when p >= 0.05. Bout duration is first to last lick (M1).

In [ ]:
rows = []
for s in PR:
    b = P.bouts(s["licks"]["reward"])
    rows.append(dict(mouse=s["subject"], bouts=len(b), mean_duration_s=(b.offset - b.onset).mean()))
d6k = pd.DataFrame(rows)
print(d6k.round(2).to_string(index=False))
r, p = stats.pearsonr(d6k.mean_duration_s, d6k.bouts)
print(f"n = {len(d6k)}, r = {r:.3f}, r2 = {r * r:.3f}, p = {p:.3g}")
ED6K = p

**Result: reproduced.** Across the 9 BLA-mPFC mice mean alcohol bout duration does not correlate with the number of bouts (r = 0.159, r2 = 0.025, p = 0.68).**

## mPFC responses on the stimulation sessions

For each mPFC session, the isolated bouts (M2) of the reward bottle (alcohol or sucrose) and of
water are labelled by their trial: stimulated choice trial (type 4) or not (types 1 to 3). Per
neuron, M3 traces are averaged over a matched number of bouts per condition (reading a random,
reading b first bouts). For the stimulation claims the conditions are stimulated and
nonstimulated bouts of one liquid; for the housing claims the conditions are the reward liquid
and water of the session, on (a) nonstimulated trials or (b) all trials, as the criteria name.
Neurons whose response is undefined (no bout in a condition) are dropped and counted.

In [ ]:
def mpfc_stim(grp, liquid, rng, how):
    """Per neuron responses, stimulated and nonstimulated bouts of `liquid`, matched per session."""
    st, ns, mouse, nb = [], [], [], []
    for s in grp:
        b = P.session_bouts(s)
        liq = (b.liquid == liquid).values
        tr, m, _ = P.condition_traces(s, b, {"stim": liq & (b.trial_type == 4).values, "nostim": liq & (b.trial_type != 4).values}, rng, how)
        st.append(P.window_mean(tr["stim"])); ns.append(P.window_mean(tr["nostim"]))
        mouse += [s["subject"]] * s["n_rois"]; nb.append(m)
    return np.concatenate(st), np.concatenate(ns), np.array(mouse), nb

def mpfc_housing(grp, liquid, rng, how, trials):
    out, mouse, nb = [], [], []
    for s in grp:
        b = P.session_bouts(s)
        sel = (b.trial_type != 4).values if trials == "nonstimulated" else np.ones(len(b), bool)
        tr, m, _ = P.condition_traces(s, b, {"reward": sel & (b.liquid == "reward").values, "water": sel & (b.liquid == "water").values}, rng, how)
        out.append(P.window_mean(tr[liquid])); mouse += [s["subject"]] * s["n_rois"]; nb.append(m)
    return np.concatenate(out), np.array(mouse), nb

MATCHED = [s["path"].split("/")[1][4:45] for s in S]
TYPES_OK = set(SC.index[SC.types_match_licks])
def subset_ok(grp):
    return [s for s in grp if s["path"].split("/")[1][4:45] in TYPES_OK]
for lab, g in (("alcohol GH", MP_A_GH), ("alcohol SI", MP_A_SI), ("sucrose GH", MP_S_GH), ("sucrose SI", MP_S_SI)):
    print(f"{lab}: sessions whose trial types match the licks: {[s['subject'] for s in subset_ok(g)]}")

## fig5b-isolation-increases-mpfc-alcohol

> Isolation increases mPFC neuron responses to alcohol.

Criterion: M3 alcohol responses of mPFC neurons in the alcohol stimulation session before and
after isolation; trials (a) nonstimulated only, (b) all; neurons are units; reproduced when, under
both, the isolated mean exceeds the group-housed mean with one-sided p < 0.05; the two-sided p is
reported.

In [ ]:
def housing_test(gh, si, liquid, seed, alt):
    res = {}
    for trials in ("nonstimulated", "all"):
        for how in ("random", "first"):
            rng = np.random.default_rng(seed)
            a, ma, na = mpfc_housing(gh, liquid, rng, how, trials)
            b, mb, nb = mpfc_housing(si, liquid, rng, how, trials)
            a, b = a[np.isfinite(a)], b[np.isfinite(b)]
            t = stats.ttest_ind(b, a)
            one = stats.ttest_ind(b, a, alternative=alt).pvalue if alt != "two-sided" else t.pvalue
            res[(trials, how)] = (a.mean(), b.mean(), one, t.pvalue)
            print(f"{liquid}, trials {trials}, bouts {how}: GH {a.mean():.3f} (n={len(a)}, dropped {len(ma) - len(a)}), "
                  f"SI {b.mean():.3f} (n={len(b)}, dropped {len(mb) - len(b)}); t = {t.statistic:.2f}, p ({alt}) = {one:.3g}, two-sided {t.pvalue:.3g}; matched bouts GH {na}, SI {nb}")
        ps = []
        for i in range(20):
            rng = np.random.default_rng(seed + 1000 * (i + 1))
            a = mpfc_housing(gh, liquid, rng, "random", trials)[0]; b = mpfc_housing(si, liquid, rng, "random", trials)[0]
            a, b = a[np.isfinite(a)], b[np.isfinite(b)]
            ps.append(stats.ttest_ind(b, a, alternative=alt).pvalue)
        print(f"   reading a over 20 further random draws, trials {trials}: median p ({alt}) {np.median(ps):.3g}, p < 0.05 in {np.mean(np.array(ps) < 0.05):.0%}")
    return res
FIG5B = housing_test(MP_A_GH, MP_A_SI, "reward", 51, "greater")
# by mouse
for lab, g in (("GH", MP_A_GH), ("SI", MP_A_SI)):
    r, m, _ = mpfc_housing(g, "reward", np.random.default_rng(51), "random", "nonstimulated")
    print(lab, "mean alcohol response by mouse:", {k: round(float(np.nanmean(r[m == k])), 3) for k in np.unique(m)})

**Result: not reproduced, resting on the reading of bout matching.** Under reading a (random matched bouts) the isolated mean does not exceed the group-housed one significantly: nonstimulated trials 0.099 against 0.096 (one-sided p = 0.49; median over 20 further draws 0.24, p < 0.05 in 10%), all trials 0.162 against 0.076 (p = 0.13; median 0.14, 0%). Under reading b (first bouts) it does: 0.172 against -0.015 (p = 0.024) and 0.161 against 0.008 (p = 0.022). The outcome rests on reading a, adopted as an assumption. Matched bout counts are 1 to 15 per mouse.**

## fig5c-isolation-no-water-change

> Isolation does not change mPFC responses to water.

Criterion: a claim of absence, the test of fig5b on water bouts, both trial readings, two-sided
unpaired t-test at p < 0.05; reproduced when p >= 0.05 under both.

In [ ]:
FIG5C = housing_test(MP_A_GH, MP_A_SI, "water", 52, "two-sided")

**Result: reproduced.** Water responses do not differ between housing conditions under any reading (two-sided p 0.50 to 0.89; over 20 further draws median 0.82 and 0.92).**

## fig5d-isolation-decreases-mpfc-sucrose

> Isolation decreases mPFC neuron responses to sucrose.

Criterion: as fig5b for sucrose bouts of the sucrose stimulation sessions, both trial readings;
reproduced when, under both, the isolated mean is below the group-housed mean with one-sided p <
0.05; the two-sided p is given.

In [ ]:
FIG5D = housing_test(MP_S_GH, MP_S_SI, "reward", 53, "less")

**Result: not reproduced.** The isolated sucrose response is not significantly below the group-housed one under any reading (one-sided p 0.22 to 0.56; isolated mean -0.025 to 0.025 against 0.016 to 0.074; over 20 further draws median p 0.36 and 0.50, never below 0.05).**

In [ ]:
def stim_test(grp, liquid, seed, alt, label):
    res = {}
    for subset, g in (("all sessions", grp), ("sessions whose types match the licks", subset_ok(grp))):
        for how in ("random", "first"):
            if not g:
                print(f"{label}, {subset}: no session"); continue
            st, ns, mouse, nb = mpfc_stim(g, liquid, np.random.default_rng(seed), how)
            ok = np.isfinite(st) & np.isfinite(ns)
            t = stats.ttest_rel(st[ok], ns[ok])
            one = stats.ttest_rel(st[ok], ns[ok], alternative=alt).pvalue if alt != "two-sided" else t.pvalue
            res[(subset, how)] = (ns[ok].mean(), st[ok].mean(), one, t.pvalue, ok.sum())
            print(f"{label}, {subset}, bouts {how}: {ok.sum()} neurons of {len(st)} from {len(g)} mice; nonstim {ns[ok].mean():.3f}, stim {st[ok].mean():.3f}, "
                  f"t({ok.sum() - 1}) = {t.statistic:.2f}, p ({alt}) = {one:.3g}, two-sided {t.pvalue:.3g}; matched bouts per mouse {nb}")
            if subset == "all sessions" and how == "random":
                print("   by mouse, stim minus nonstim (neurons):", {m: f"{np.mean(st[ok & (mouse == m)] - ns[ok & (mouse == m)]):.3f} ({(ok & (mouse == m)).sum()})" for m in np.unique(mouse)})
        if subset == "all sessions" and g:
            ps = []
            for i in range(20):
                st, ns, mouse, nb = mpfc_stim(g, liquid, np.random.default_rng(seed + 1000 * (i + 1)), "random")
                ok = np.isfinite(st) & np.isfinite(ns)
                ps.append(stats.ttest_rel(st[ok], ns[ok], alternative=alt).pvalue)
            print(f"   reading a over 20 further random draws: median p ({alt}) {np.median(ps):.3g}, p < 0.05 in {np.mean(np.array(ps) < 0.05):.0%}")
    return res

def proportion_test(grp, liquid, seed, label):
    """Per mouse percentage excited and inhibited (M4 threshold), stimulated against nonstimulated; paired over mice."""
    res = {}
    for subset, g in (("all sessions", grp), ("sessions whose types match the licks", subset_ok(grp))):
        for how in ("random", "first"):
            if len(g) < 2:
                print(f"{label}, {subset}: fewer than 2 sessions"); continue
            st, ns, mouse, nb = mpfc_stim(g, liquid, np.random.default_rng(seed), how)
            rows = []
            for m in np.unique(mouse):
                k = (mouse == m) & np.isfinite(st) & np.isfinite(ns)
                if k.sum() == 0:
                    continue
                rows.append(dict(mouse=m, n=k.sum(), exc_stim=100 * np.mean(st[k] >= P.Z_THRESHOLD), exc_nostim=100 * np.mean(ns[k] >= P.Z_THRESHOLD),
                                 inh_stim=100 * np.mean(st[k] <= -P.Z_THRESHOLD), inh_nostim=100 * np.mean(ns[k] <= -P.Z_THRESHOLD)))
            d = pd.DataFrame(rows)
            for kind_ in ("exc", "inh"):
                t = stats.ttest_rel(d[f"{kind_}_stim"], d[f"{kind_}_nostim"])
                lo = stats.ttest_rel(d[f"{kind_}_stim"], d[f"{kind_}_nostim"], alternative="less").pvalue
                res[(subset, how, kind_)] = (d[f"{kind_}_nostim"].mean(), d[f"{kind_}_stim"].mean(), t.pvalue, lo, len(d))
                print(f"{label}, {subset}, bouts {how}, {kind_}: {len(d)} mice; nonstim {d[f'{kind_}_nostim'].mean():.1f}%, stim {d[f'{kind_}_stim'].mean():.1f}%, "
                      f"t({len(d) - 1}) = {t.statistic:.2f}, two-sided p = {t.pvalue:.3g}, one-sided (stim lower) p = {lo:.3g}")
            if how == "random" and subset == "all sessions":
                print(d.round(1).to_string(index=False))
        if subset == "all sessions" and len(g) >= 2:
            ps = {"exc": [], "inh": []}
            for i in range(20):
                st, ns, mouse, nb = mpfc_stim(g, liquid, np.random.default_rng(seed + 1000 * (i + 1)), "random")
                for kind_ in ps:
                    thr = (lambda x: x >= P.Z_THRESHOLD) if kind_ == "exc" else (lambda x: x <= -P.Z_THRESHOLD)
                    a_, b_ = [], []
                    for m in np.unique(mouse):
                        k = (mouse == m) & np.isfinite(st) & np.isfinite(ns)
                        if k.sum():
                            a_.append(100 * np.mean(thr(st[k]))); b_.append(100 * np.mean(thr(ns[k])))
                    ps[kind_].append(stats.ttest_rel(a_, b_).pvalue)
            print("   reading a over 20 further random draws, two-sided p: " + "; ".join(
                f"{k} median {np.nanmedian(v):.3g}, p < 0.05 in {np.mean(np.array(v) < 0.05):.0%}" for k, v in ps.items()))
    return res

## fig6c-stim-increases-mpfc-alcohol-gh

> In group-housed mice, BLA-mPFC stimulation increases mPFC responses to alcohol, mimicking
> isolation.

Criterion: M3 alcohol responses of each neuron on stimulated and nonstimulated choice trials of
the group-housed alcohol stimulation session; paired over neurons; reproduced when stimulated
exceeds nonstimulated with one-sided p < 0.05; the two-sided p is reported. Nonstimulated bouts are
those of trials without light (types 1 to 3).

In [ ]:
FIG6C = stim_test(MP_A_GH, "reward", 61, "greater", "alcohol GH")

**Result: not reproduced.** In the group-housed alcohol session stimulated responses (0.097) are not significantly above nonstimulated ones (0.048; one-sided p = 0.34, reading b 0.30, over 20 further draws median 0.19, never below 0.05), over 605 neurons with both responses from 6 mice; matched bouts are 0 to 8 per mouse. On the 3 sessions whose trial types match the licks the stimulated mean is lower (0.107 against 0.188).**

## fig6de-stim-fewer-alcohol-inhibited

> BLA-mPFC stimulation lowers the proportion of mPFC neurons inhibited by alcohol, without
> changing the proportion excited.

Criterion: per mouse, the percentage of its neurons inhibited (and excited) per M4 on stimulated
and nonstimulated trials, group-housed, paired over mice; reproduced when the inhibited proportion
falls with p < 0.05 and the excited proportion does not change at p < 0.05, under both sidedness
readings.

In [ ]:
FIG6DE = proportion_test(MP_A_GH, "reward", 62, "alcohol GH")

**Result: not reproduced.** The inhibited proportion does not fall significantly with stimulation (10.1% to 9.2%, two-sided p = 0.25, one-sided 0.12; reading b 12.1% to 9.2%, p = 0.47; over 20 further draws median p 0.54), and the excited proportion does not change (p = 0.30 and 0.87), over 5 mice (the sixth has no stimulated alcohol bout).**

## fig6f-stim-decreases-mpfc-alcohol-si

> After isolation, BLA-mPFC stimulation decreases mPFC responses to alcohol.

Criterion: as fig6c in the isolation session; reproduced when stimulated is below nonstimulated
with one-sided p < 0.05; the two-sided p is reported.

In [ ]:
FIG6F = stim_test(MP_A_SI, "reward", 63, "less", "alcohol SI")

**Result: not reproduced.** In the isolation alcohol session stimulated responses (0.172) are not below nonstimulated ones (0.104; one-sided p = 0.78; reading b 0.144 against 0.133, p = 0.55; median over 20 further draws 0.77), over 423 neurons from 5 mice.**

## fig6gj-stim-no-water-effect

> BLA-mPFC stimulation does not change mPFC responses to water, nor the proportions of excited or
> inhibited water neurons, before or after isolation.

Criterion: a claim of absence on the tests of fig6c (paired over neurons) and fig6de (paired over
mice) on water bouts, both housing conditions, two-sided, p < 0.05; reproduced when none reaches
p < 0.05. The alcohol sessions are used.

In [ ]:
FIG6GJ = {}
for lab, g, seed in (("water GH", MP_A_GH, 64), ("water SI", MP_A_SI, 65)):
    FIG6GJ[(lab, "resp")] = stim_test(g, "water", seed, "two-sided", lab)
    FIG6GJ[(lab, "prop")] = proportion_test(g, "water", seed, lab)

**Result: reproduced.** No water test reaches p < 0.05: responses over neurons, group-housed p = 0.44 and 0.64, isolated p = 0.12 and 0.36; proportions over mice, group-housed excited p = 0.39 and 0.82, inhibited p = 0.72 and 0.81, isolated (4 mice) excited p = 0.13 and 0.78, inhibited p = 0.76 and 0.46.**

## fig6k-stim-decreases-mpfc-sucrose-gh

> In group-housed mice, BLA-mPFC stimulation decreases mPFC responses to sucrose, mimicking
> isolation.

Criterion: as fig6c on sucrose bouts of the group-housed sucrose stimulation session; reproduced
when stimulated is below nonstimulated with one-sided p < 0.05.

In [ ]:
FIG6K = stim_test(MP_S_GH, "reward", 66, "less", "sucrose GH")

**Result: not reproduced.** In the group-housed sucrose session stimulated responses are not significantly below nonstimulated ones (reading a 0.040 against 0.025, one-sided p = 0.63; reading b 0.040 against 0.099, p = 0.11; median over 20 further draws 0.58), over 925 neurons from 8 mice.**

## fig6lm-stim-fewer-sucrose-excited

> BLA-mPFC stimulation lowers the proportion of mPFC neurons excited by sucrose, without changing
> the proportion inhibited.

Criterion: as fig6de on sucrose, group-housed; reproduced when the excited proportion falls with
p < 0.05 and the inhibited proportion does not change at p < 0.05; both sidedness readings.

In [ ]:
FIG6LM = proportion_test(MP_S_GH, "reward", 67, "sucrose GH")

**Result: not reproduced, resting on the reading of bout matching.** Under reading a the excited proportion does not fall (4.2% to 4.1%, two-sided p = 0.89; median over 20 further draws 0.35, below 0.05 in 5%). Under reading b it falls from 6.2% to 4.1% with one-sided p = 0.036 (two-sided 0.071), and the inhibited proportion does not change (p = 0.12), which meets the criterion under the one-sided reading only. The outcome rests on reading a.**

## fig6n-isolation-occludes-sucrose-effect

> After isolation, BLA-mPFC stimulation no longer changes mPFC responses to sucrose.

Criterion: a claim of absence on the paired t-test over neurons, stimulated against
nonstimulated in the isolation sucrose session, two-sided at p < 0.05; reproduced when p >= 0.05.

In [ ]:
FIG6N = stim_test(MP_S_SI, "reward", 68, "two-sided", "sucrose SI")

**Result: reproduced.** In the isolation sucrose session stimulation does not change sucrose responses (two-sided p = 0.47, reading b 0.12; median over 20 further draws 0.45), over 543 neurons from 6 mice.**

## ed8n-stim-fewer-sucrose-inhibited-si

> During isolation, BLA-mPFC stimulation lowers the proportion of mPFC neurons inhibited by
> sucrose, with no change in those excited.

Criterion: as fig6lm in the isolation sucrose session, paired over mice, two-sided; reproduced
when the inhibited proportion falls with p < 0.05 and the excited proportion does not change at
p < 0.05.

In [ ]:
ED8N = proportion_test(MP_S_SI, "reward", 69, "sucrose SI")

**Result: not reproduced, resting on the reading of bout matching.** The inhibited proportion is lower on stimulated trials (1.9% to 0.6% with the first random draw, two-sided p = 0.048, with no change in the excited proportion, p = 0.64), but this p is not stable: over 20 further random draws the median p is 0.092 and only 5% fall below 0.05, and reading b gives p = 0.18. The fall is not significant once the draw is not chosen; the inhibited proportions are 0% to 4.6% per mouse.**

## ed8op-stim-no-alcohol-proportions-si

> During isolation, BLA-mPFC stimulation does not change the proportions of mPFC neurons excited
> or inhibited by alcohol.

Criterion: a claim of absence; two-sided paired t-test over the 5 mice of the isolation alcohol
session, percentages excited and inhibited per M4, stimulated against nonstimulated, p < 0.05;
reproduced when neither reaches p < 0.05.

In [ ]:
ED8OP = proportion_test(MP_A_SI, "reward", 70, "alcohol SI")

**Result: reproduced.** Neither proportion changes with stimulation in the isolation alcohol session (excited p = 0.83 and 0.76, inhibited p = 0.65 and 0.69 under readings a and b; over 20 further draws median p 0.74 and 0.65), over 5 mice.**

## Summary

| Claim | Outcome |
|---|---|
| ed5d-cue-response-modest | not reproduced (assumed: cohort to population) |
| fig3f-blampfc-distinct-clusters | reproduced (assumed: cohort to population) |
| ed5ef-alcohol-water-nonoverlapping | reproduced (assumed: cohort to population) |
| fig3h-nonspecific-decoding-85, fig3h-blampfc-decodes-better, ed6b-decoding-above-chance | blocked: too few decoding bouts per mouse |
| fig3k-blampfc-response-predicts-bouts | not reproduced (assumed: cohort to population) |
| fig3ijl-no-other-correlations | reproduced (assumed: cohort to population) |
| ed6ij-cue-responses-no-correlation | not reproduced (assumed: cohort to population) |
| ed6k-bout-duration-not-number | reproduced (assumed: cohort to population) |
| fig5b-isolation-increases-mpfc-alcohol | not reproduced (assumed: random bout matching) |
| fig5c-isolation-no-water-change | reproduced |
| fig5d-isolation-decreases-mpfc-sucrose | not reproduced |
| fig6c-stim-increases-mpfc-alcohol-gh | not reproduced |
| fig6de-stim-fewer-alcohol-inhibited | not reproduced |
| fig6f-stim-decreases-mpfc-alcohol-si | not reproduced |
| fig6gj-stim-no-water-effect | reproduced |
| fig6k-stim-decreases-mpfc-sucrose-gh | not reproduced |
| fig6lm-stim-fewer-sucrose-excited | not reproduced (assumed: random bout matching) |
| fig6n-isolation-occludes-sucrose-effect | reproduced |
| ed8n-stim-fewer-sucrose-inhibited-si | not reproduced (assumed: random bout matching) |
| ed8op-stim-no-alcohol-proportions-si | reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

How the analysis differs from the paper's: every CNMF-E ROI is used, as the curated neurons are not
in the dandiset (1113 nonspecific ROIs against the paper's 540, 347 BLA-mPFC against 196, 748 and
439 mPFC alcohol ROIs against 240 and 281); the populations are assigned to cohorts by their mouse
counts; z-scores are left undefined where the baseline s.d. is under 5% of the session s.d.; the
files of three sessions whose traces belong to another session are left out; matched bout counts
are small (often 1 to 10 per mouse); trial types (and so stimulation) are as stored, although in
many sessions they do not match the licks; the sucrose groups hold 8 and 6 mice (paper 7 and 7).
The decoding analyses could not be run.

What was found about the dataset: all 53 cued-2BC files carry the same cue times and one of two
trial-type sequences, and in every session of 5 of the 9 mPFC mice and in 10 of 11 cohort-16 sessions
there are licks on the bottle a forced trial does not offer; three pairs of files hold identical imaging with different
licks (cohort 16 M1-4 and M2-1, M3-3 and M3-4; cohort 13b M2-2 and M3-2 isolation sucrose); the
files hold uncurated ROIs; nothing in the files says which BLA cohort is projection-specific (the
virus field is empty); cohort-9 traces are nonnegative with exact zeros, while the Methods say
CNMF-E was run without nonnegativity; four mPFC sessions hold 44791 to 65535 frames at 10 Hz (75
to 109 min) for sessions whose last lick is before 3900 s; the alignment of traces and licks could not
be shown for 13 of 53 sessions (few bouts or no response peak near lag 0: cohort 13b M1-1 group-housed and
isolated alcohol, M1-2 isolated sucrose, M2-1 group-housed and isolated alcohol, M2-2 group-housed sucrose,
M3-2 isolated sucrose; cohort 16 M1-2, M2-1, M3-1, M3-2; cohort 9 M5-1, M5-2); the resident-intruder files hold no pose
tracking or interaction annotation.

In [ ]:
print(f"runtime {(time.time() - t_start) / 60:.1f} min")